In [ ]:
from pyspark.sql import functions as F

# Paths
path_root = "/Volumes/transientes_pd/d5f1a1f/c1331207/EstudoNPBB/gold"

df_spend_window = spark.read.format("delta").load(f"{path_root}/gold_spend_event_window")
df_cohort = spark.read.format("delta").load(f"{path_root}/gold_spend_cohort")

df_spend_window.createOrReplaceTempView("gold_spend_event_window")
df_cohort.createOrReplaceTempView("gold_spend_cohort")


In [ ]:
# Spend total por janela, filtrável por event_id/event_name no display.
spend_window_long = df_spend_window.selectExpr(
    "event_id",
    "event_name",
    "event_location",
    "event_date",
    "person_key",
    "stack(4, 'spend_pre_30', spend_pre_30, 'spend_0_30', spend_0_30, 'spend_31_60', spend_31_60, 'spend_61_90', spend_61_90) as (window, spend)"
)

display(
    spend_window_long
    .groupBy("event_id", "event_name", "event_location", "event_date", "window")
    .agg(
        F.sum("spend").alias("total_spend"),
        F.avg("spend").alias("avg_spend_por_pessoa"),
        F.countDistinct("person_key").alias("n_pessoas")
    )
    .orderBy("event_date", "event_id", "window")
)


In [ ]:
# Cohorts de ativação por evento e semana.
display(
    df_cohort
    .select(
        "event_id",
        "event_name",
        "event_location",
        "event_date",
        "cohort_week",
        "n_cards",
        "cards_activated_30",
        "cards_activated_60",
        "activation_rate_30",
        "activation_rate_60",
    )
    .orderBy("event_date", "event_id", "cohort_week")
)


In [ ]:
%sql
SELECT
  event_id,
  event_name,
  event_location,
  event_date,
  cohort_week,
  n_cards,
  activation_rate_30,
  activation_rate_60
FROM gold_spend_cohort
ORDER BY event_date, event_id, cohort_week
